# Stage 1 — positive weighting trên Colab T4
Notebook này chạy 12 validation experiments, tự resume và lưu phân tích lên Drive. External test luôn khóa trong Stage 1.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/HuyDuongQuoc/DATH-HK261-explainable-cxr.git"
REPO_BRANCH = "update_data"
REPO_DIR = Path("/content/DATH-HK261-explainable-cxr")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive" / "DATH_CXR"
TRAIN_ARCHIVE = DRIVE_ROOT / "datasets" / "consensus_v1_224.zip"
EXPERIMENT_GROUP = "aug_auprc_v1"
NOTEBOOK_GROUP = "03_testing"
STAGE_ID = "stage1_pos_weight_v1"
CHECKPOINT_ROOT = DRIVE_ROOT / "checkpoints" / EXPERIMENT_GROUP / NOTEBOOK_GROUP / STAGE_ID
RUN_LOG_ROOT = DRIVE_ROOT / "run_logs" / EXPERIMENT_GROUP / NOTEBOOK_GROUP / STAGE_ID
HARDWARE_ROOT = RUN_LOG_ROOT / "hardware"
LOCAL_DATA_ROOT = Path("/content/datasets")
TRAIN_PROCESSED_ROOT = LOCAL_DATA_ROOT / "consensus_v1_224"
PROTOCOL_ROOT = REPO_DIR / "data" / "protocols" / "vindr_s8515_exttest_v1"
RUNTIME_CONFIG_ROOT = Path("/content/runtime_configs") / STAGE_ID
EXPORT_PATH = Path("/content/exports/03_testing_stage1_pos_weight_v1_results.zip")
RUN_FINAL_TEST = False
FINAL_SELECTION = []
REQUIRE_T4 = True
ACTIVE_MODELS = ["densenet121", "resnet50", "vit_b_16"]
ACTIVE_WEIGHTINGS = ["raw", "sqrt", "sqrtcap5", "sqrtcap3"]
MAX_NEW_RUNS_PER_SESSION = None
EXPECTED_IMAGES = 15_000
EXPECTED_MASKS = 15_000
EXPECTED_TRAIN_ROWS = 12_750
EXPECTED_VAL_ROWS = 2_250

MODEL_SPECS = [
    ("densenet121", "configs/densenet121_aug_auprc.yaml"),
    ("resnet50", "configs/resnet50_aug_auprc.yaml"),
    ("vit_b_16", "configs/vit_aug_auprc.yaml"),
]
WEIGHTING_SPECS = {
    "raw": {"mode": "raw_ratio", "max_value": None},
    "sqrt": {"mode": "sqrt_ratio", "max_value": None},
    "sqrtcap5": {"mode": "sqrt_ratio_capped", "max_value": 5.0},
    "sqrtcap3": {"mode": "sqrt_ratio_capped", "max_value": 3.0},
}
MODEL_RUN_NAMES = {"densenet121": "densenet121", "resnet50": "resnet50", "vit_b_16": "vit"}
EXPERIMENTS = [
    {
        "model_name": model, "config": config, "weighting": weighting,
        "run_name": f"{MODEL_RUN_NAMES[model]}_stage1_bce_{weighting}_aug_auprc_s8515_exttest_v1_seed42_amp",
        "positive_weight": WEIGHTING_SPECS[weighting],
    }
    for model, config in MODEL_SPECS for weighting in WEIGHTING_SPECS
]
assert len(EXPERIMENTS) == 12 and len({item["run_name"] for item in EXPERIMENTS}) == 12
print([item["run_name"] for item in EXPERIMENTS])

In [ ]:
from google.colab import drive
drive.mount(str(DRIVE_MOUNT), force_remount=False)
if not TRAIN_ARCHIVE.is_file():
    raise FileNotFoundError(f"Missing train archive: {TRAIN_ARCHIVE}")
for path in (CHECKPOINT_ROOT, RUN_LOG_ROOT, HARDWARE_ROOT):
    path.mkdir(parents=True, exist_ok=True)

In [ ]:
import base64, os, subprocess, sys
from getpass import getpass
github_token = getpass("GitHub PAT (leave blank for public repo): " ).strip()
git_env = os.environ.copy()
if github_token:
    auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    git_env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader", "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}"})
def run_git(args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, env=git_env, check=True)
if REPO_DIR.exists():
    run_git(["fetch", "origin", REPO_BRANCH], REPO_DIR)
    run_git(["checkout", REPO_BRANCH], REPO_DIR)
    run_git(["pull", "--ff-only", "origin", REPO_BRANCH], REPO_DIR)
else:
    run_git(["clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
commit_sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True, capture_output=True, text=True).stdout.strip()
github_token = auth = None
print("Repository commit:", commit_sha)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-common.txt")], check=True)
import json, platform, torch
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"], check=True, capture_output=True, text=True).stdout.strip()
gpu_name, total_memory_mib, driver = [part.strip() for part in smi.split(",")]
if REQUIRE_T4 and "T4" not in gpu_name.upper(): raise RuntimeError(f"Expected T4, got {gpu_name}")
if float(total_memory_mib) < 15000: raise RuntimeError(f"Expected at least 15 GB VRAM, got {total_memory_mib} MiB")
with torch.autocast(device_type="cuda", dtype=torch.float16):
    probe = torch.ones(8, 8, device="cuda") @ torch.ones(8, 8, device="cuda")
if not torch.isfinite(probe).all(): raise RuntimeError("AMP FP16 probe failed")
hardware_profile = {
    "gpu_name": gpu_name, "total_memory_mib": float(total_memory_mib), "driver": driver,
    "cpu_count": os.cpu_count(), "platform": platform.platform(), "torch": torch.__version__,
    "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
    "batch_size": 8, "validation_batch_size": 64, "num_workers": 2, "prefetch_factor": 2,
    "amp": True, "cudnn_benchmark": False, "cudnn_deterministic": True,
}
(HARDWARE_ROOT / "hardware_profile.json").write_text(json.dumps(hardware_profile, indent=2), encoding="utf-8")
print(hardware_profile); print("AMP FP16 probe: OK")

In [ ]:
import shutil, tempfile, zipfile
import pandas as pd
from src.data.protocol_splitting import assignment_sha256
from src.utils.checksums import file_sha256
def safe_extract(archive_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            try: target.relative_to(destination)
            except ValueError as error: raise ValueError(f"Unsafe ZIP member: {member.filename}") from error
        archive.extractall(destination)
def validate_train_data(root):
    required = [root / "images", root / "masks", root / "manifest.csv", root / "metadata.json"]
    if not all(path.exists() for path in required): raise FileNotFoundError(root)
    train = pd.read_csv(PROTOCOL_ROOT / "train_manifest.csv", dtype={"image_id": str})
    val = pd.read_csv(PROTOCOL_ROOT / "val_manifest.csv", dtype={"image_id": str})
    manifest = pd.read_csv(root / "manifest.csv", dtype={"image_id": str})
    if len(train) != EXPECTED_TRAIN_ROWS or len(val) != EXPECTED_VAL_ROWS: raise ValueError("Split row mismatch")
    if set(train.image_id) & set(val.image_id): raise ValueError("Train/validation overlap")
    if set(train.image_id) | set(val.image_id) != set(manifest.image_id): raise ValueError("Split coverage mismatch")
    metadata = json.loads((PROTOCOL_ROOT / "split_metadata.json").read_text(encoding="utf-8"))
    if assignment_sha256({"train": train, "val": val}) != metadata["assignment_sha256"]: raise ValueError("Split fingerprint mismatch")
    if len(list((root / "images").glob("*.png"))) != EXPECTED_IMAGES: raise ValueError("PNG count mismatch")
    if len(list((root / "masks").glob("*.npz"))) != EXPECTED_MASKS: raise ValueError("NPZ count mismatch")
    return {"split_assignment_sha256": metadata["assignment_sha256"], "validation_manifest_sha256": file_sha256(PROTOCOL_ROOT / "val_manifest.csv")}
if not TRAIN_PROCESSED_ROOT.exists():
    temporary = Path(tempfile.mkdtemp(prefix="stage1-data-", dir="/content"))
    try:
        safe_extract(TRAIN_ARCHIVE, temporary)
        extracted = temporary / "consensus_v1_224"
        validate_train_data(extracted)
        TRAIN_PROCESSED_ROOT.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(extracted), str(TRAIN_PROCESSED_ROOT))
    finally: shutil.rmtree(temporary, ignore_errors=True)
train_summary = validate_train_data(TRAIN_PROCESSED_ROOT)
print(train_summary)

In [ ]:
import csv, time, yaml
from src.analysis.stage1 import analyze_stage1
from scripts.train import build_augmentation_protocol, build_training_protocol
from src.utils.config import load_config
(RUN_LOG_ROOT / "experiment_registry.json").write_text(json.dumps(EXPERIMENTS, indent=2), encoding="utf-8")
def build_runtime_config(experiment):
    config = load_config(REPO_DIR / experiment["config"])
    config["data"]["processed_root"] = str(TRAIN_PROCESSED_ROOT)
    config["data"]["protocol_dir"] = str(PROTOCOL_ROOT)
    config["data"]["train_manifest_path"] = str(PROTOCOL_ROOT / "train_manifest.csv")
    config["data"]["val_manifest_path"] = str(PROTOCOL_ROOT / "val_manifest.csv")
    config["training"].update({"batch_size": 8, "validation_batch_size": 64, "num_workers": 2, "persistent_workers": True, "prefetch_factor": 2, "show_progress": False, "monitor": "macro_average_precision"})
    config["training"]["loss"] = {"name": "weighted_bce", "positive_weight": experiment["positive_weight"]}
    if not config["preprocessing"]["use_augmentation"]: raise ValueError("Stage 1 requires augmentation")
    runtime_dir = RUNTIME_CONFIG_ROOT / experiment["run_name"]
    runtime_dir.mkdir(parents=True, exist_ok=True)
    path = runtime_dir / "default.yaml"
    path.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")
    load_config(path)
    return path, config
def run_complete(experiment, config):
    path = CHECKPOINT_ROOT / experiment["run_name"] / "last.pt"
    if not path.is_file(): return False, False
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    expected_loss = {"name": "weighted_bce", "positive_weight": experiment["positive_weight"]}
    expected_training = build_training_protocol(config["training"], build_augmentation_protocol(config["preprocessing"]), expected_loss, amp_enabled=True)
    if checkpoint.get("model_name") != experiment["model_name"] or checkpoint.get("loss_protocol") != expected_loss or checkpoint.get("training_protocol") != expected_training: raise ValueError(f"Checkpoint protocol mismatch: {path}")
    complete = int(checkpoint["epoch"]) + 1 >= config["training"]["epochs"] or int(checkpoint["bad_epochs"]) >= config["training"]["patience"]
    return complete, not complete
def validate_runtime_matrix():
    runtime = {}
    normalized = {}
    for experiment in EXPERIMENTS:
        path, config = build_runtime_config(experiment)
        runtime[experiment["run_name"]] = (path, config)
        comparable = json.loads(json.dumps(config))
        comparable["training"]["loss"]["positive_weight"] = "<variant>"
        model = experiment["model_name"]
        if model in normalized and normalized[model] != comparable: raise ValueError(f"Runtime configs differ beyond positive weighting: {model}")
        normalized[model] = comparable
    return runtime
RUNTIME_CONFIGS = validate_runtime_matrix()
def start_telemetry(run_name):
    path = HARDWARE_ROOT / f"gpu_telemetry_{run_name}.csv"
    handle = path.open("w", encoding="utf-8", newline="")
    handle.write("timestamp,utilization_gpu_percent,utilization_memory_percent,memory_used_mib,power_draw_watts\n"); handle.flush()
    process = subprocess.Popen(["nvidia-smi", "--query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,power.draw", "--format=csv,noheader,nounits", "--loop-ms=5000"], stdout=handle, stderr=subprocess.DEVNULL, text=True)
    return process, handle
def stop_telemetry(process, handle):
    process.terminate()
    try: process.wait(timeout=10)
    except subprocess.TimeoutExpired: process.kill()
    handle.close()

In [ ]:
new_runs = 0
for index, experiment in enumerate(EXPERIMENTS, start=1):
    if experiment["model_name"] not in ACTIVE_MODELS or experiment["weighting"] not in ACTIVE_WEIGHTINGS: continue
    config_path, config = RUNTIME_CONFIGS[experiment["run_name"]]
    checkpoint_dir = CHECKPOINT_ROOT / experiment["run_name"]
    result_dir = RUN_LOG_ROOT / experiment["run_name"]
    checkpoint_dir.mkdir(parents=True, exist_ok=True); result_dir.mkdir(parents=True, exist_ok=True)
    complete, resume = run_complete(experiment, config)
    if complete:
        print(f"[{index}/12] skip complete: {experiment['run_name']}")
    else:
        if MAX_NEW_RUNS_PER_SESSION is not None and new_runs >= MAX_NEW_RUNS_PER_SESSION: break
        command = [sys.executable, str(REPO_DIR / "scripts" / "train.py"), "--config", str(config_path), "--checkpoint-dir", str(checkpoint_dir), "--result-dir", str(result_dir)]
        if resume: command.extend(["--resume", str(checkpoint_dir / "last.pt")])
        env = os.environ.copy(); env["PYTHONPATH"] = str(REPO_DIR)
        telemetry_process = telemetry_handle = None
        try:
            telemetry_process, telemetry_handle = start_telemetry(experiment["run_name"])
            subprocess.run(command, cwd=REPO_DIR, env=env, check=True)
        finally:
            if telemetry_process is not None: stop_telemetry(telemetry_process, telemetry_handle)
        new_runs += 1
    analysis = analyze_stage1(EXPERIMENTS, RUN_LOG_ROOT, CHECKPOINT_ROOT)
    display(analysis["macro"])
print("Stage 1 training pass finished.")

In [ ]:
analysis = analyze_stage1(EXPERIMENTS, RUN_LOG_ROOT, CHECKPOINT_ROOT)
display(analysis["macro"]); display(analysis["ranking"])
performance_columns = ["run_name", "model", "weighting", "train_hours", "mean_train_images_per_second", "mean_gpu_utilization", "peak_gpu_memory_allocated_gb", "peak_gpu_memory_reserved_gb"]
if not analysis["macro"].empty:
    analysis["macro"][performance_columns].to_csv(HARDWARE_ROOT / "stage1_performance_summary.csv", index=False)
if RUN_FINAL_TEST:
    if not FINAL_SELECTION: raise RuntimeError("FINAL_SELECTION must be explicitly locked before external test")
    raise RuntimeError("External test is intentionally disabled in Stage 1; use the final tuned stage notebook")
print("External test locked.")

In [ ]:
from datetime import datetime, timezone
manifest = {"created_at_utc": datetime.now(timezone.utc).isoformat(), "repository_url": REPO_URL, "repository_branch": REPO_BRANCH, "git_commit": commit_sha, "experiment_group": EXPERIMENT_GROUP, "notebook_group": NOTEBOOK_GROUP, "stage_id": STAGE_ID, "data_protocol_id": "vindr_s8515_exttest_v1", "split_assignment_sha256": train_summary["split_assignment_sha256"], "validation_manifest_sha256": train_summary["validation_manifest_sha256"], "selection_metric": "macro_average_precision", "external_test_used": False, "checkpoint_included_in_export": False}
(RUN_LOG_ROOT / "experiment_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
staging = Path(tempfile.mkdtemp(prefix="stage1-export-", dir="/content"))
try:
    destination = staging / "outputs" / EXPERIMENT_GROUP / NOTEBOOK_GROUP / STAGE_ID
    shutil.copytree(RUN_LOG_ROOT, destination)
    EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    if EXPORT_PATH.exists(): EXPORT_PATH.unlink()
    shutil.make_archive(str(EXPORT_PATH.with_suffix("")), "zip", root_dir=staging)
    with zipfile.ZipFile(EXPORT_PATH) as archive: members = archive.namelist()
    forbidden = (".pt", ".pth", ".ckpt", ".npz", ".zip")
    if any(name.lower().endswith(forbidden) for name in members): raise RuntimeError("Export contains forbidden binary artifacts")
    print("Export ready:", EXPORT_PATH)
finally: shutil.rmtree(staging, ignore_errors=True)